In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

X_well = df.drop(columns=['Metadata_well', 'phase'])
y_well = df['Metadata_well']

X_train_well, X_test_well, y_train_well, y_test_well = train_test_split(X_well, y_well, test_size=0.2, random_state=949, stratify=y_well)

In [18]:
# Define MLP model
mlp = MLPClassifier(max_iter=4000, learning_rate = 'adaptive', random_state=949)

In [23]:
# Hyperparameter tuning
param_grid = {
    'hidden_layer_sizes': [
        (22,),         # 1 hidden layer
        (11,),        # 1 hidden layer
        (22, 11),     # 2 hidden layers
        (22, 22),       # 2 hidden layers
        (22,22,22)   # 3 hidden layers
    ],
    'learning_rate_init': [0.001],
    'alpha': [0.0001, 0.001]
}
# GridSearchCV
grid_search = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

# Fit model
grid_search.fit(X_train_well, y_train_well)

# Output best parameters and best accuracy
print("Best parameters:", grid_search.best_params_)

Best parameters: {'alpha': 0.001, 'hidden_layer_sizes': (22, 22, 22), 'learning_rate_init': 0.001}


In [8]:
# Convert the cv_results_ dictionary to a DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

# Select and display relevant columns
print(
    results_df[
        [
            'param_hidden_layer_sizes',
            'mean_test_score',
            'std_test_score',
            'rank_test_score'
        ]
    ].sort_values(by='rank_test_score')
)

  param_hidden_layer_sizes  mean_test_score  std_test_score  rank_test_score
3                 (22, 22)         0.744443        0.007346                1
4             (22, 22, 22)         0.741032        0.006161                2
2                 (22, 11)         0.740606        0.007866                3
0                    (22,)         0.735199        0.005549                4
1                    (11,)         0.710044        0.006825                5


In [3]:
# Retrain with best parameters
mlp = MLPClassifier(max_iter=4000, random_state=949, hidden_layer_sizes=(22,22,22), learning_rate = 'adaptive', learning_rate_init= 0.001, alpha = 0.001)
mlp.fit(X_train_well, y_train_well)

# Predictions
y_train_pred = mlp.predict(X_train_well)
y_test_pred = mlp.predict(X_test_well)

In [4]:
# Evaluation
print("=== Training Set ===")
print("Overall Accuracy:", accuracy_score(y_train_well, y_train_pred))

print("\n=== Test Set ===")
print("Overall Accuracy:", accuracy_score(y_test_well, y_test_pred))

=== Training Set ===
Overall Accuracy: 0.7632604019301952

=== Test Set ===
Overall Accuracy: 0.7506394853112162


In [5]:
print("Test Confusion Matrix")
print(confusion_matrix(y_test_well, y_test_pred))

Test Confusion Matrix
[[1186  368  367   94   58]
 [ 485 1161  312  137   40]
 [ 293  249 1866   85  117]
 [  48   34   63 2829  164]
 [  41   26   54  182 2642]]


In [4]:
from sklearn.metrics import cohen_kappa_score
kappa = cohen_kappa_score(y_test_well, y_test_pred)
print(f"Minipatch model Cohen's kappa: {kappa:.3f}")

Minipatch model Cohen's kappa: 0.686


In [5]:
# save cohen results
# === Load existing results ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/cancer_cohen_results.csv", index_col=0)

kappa = kappa = cohen_kappa_score(y_test_well, y_test_pred)

# === Insert values ===
model_name = "MLP"
results_df.loc[model_name, "Cohen's Kappa"] = kappa

# === Save updated file ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/cancer_cohen_results.csv")

In [26]:
# save results
# === Load existing results ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/cancer_baseline_results.csv", index_col=0)

# === Compute accuracy ===
from sklearn.metrics import accuracy_score

overall_acc = accuracy_score(y_test_well, y_test_pred)

df_test = pd.DataFrame({'true': y_test_well, 'pred': y_test_pred})

# === Insert values ===
model_name = "MLP"
results_df.loc[model_name, "Accuracy"] = overall_acc

# === Save updated file ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/cancer_baseline_results.csv")